# 1. Download dos Datasets

Este notebook é responsável pela etapa de extração de dados brutos (Raw Data).
Os conjuntos de dados são baixados de suas respectivas fontes (Roboflow e Hugging Face) e armazenados no diretório `data/raw/` (dados brutos) sem sofrer nenhuma alteração ou normalização.

## Configurações Iniciais

In [1]:
import sys
from pathlib import Path
from typing import List

# Adicione a raiz do projeto ao caminho para importação de módulos
sys.path.append("..")
from src import config, datasets

print(f"Ambiente executável Python: {sys.executable}")
print(f"Diretório de dados brutos(raw): {config.RAW_DIR}")

Ambiente executável Python: /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/.venv/bin/python
Diretório de dados brutos(raw): /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/raw


## 1.1 Extração do Roboflow

Realiza a conexão com a API do Roboflow para baixar os datasets relacionados ao interior dos ônibus. 
Os dados são alocados diretamente na pasta `raw` utilizando o mapeamento configurado no arquivo `datasets.py`.

In [2]:
def download_roboflow_data() -> None:
    f"""
    Faz o download dos conjuntos de dados especificados do Roboflow diretamente para o diretório raw.
    Itera sobre os nomes dos conjuntos de dados definidos e os busca por meio do módulo de conjuntos de dados personalizados.
    """
    dataset_stage: str = "raw"
    force_download: bool = False

    roboflow_datasets: List[str] = [
        "inside-bus-view",
        "passenger-deakin",
        "passenger-detection-bus",
    ]

    print("Iniciando o processo de download dos conjuntos de dados do Roboflow...")
    datasets.download_selected(
        stage=dataset_stage,
        names=roboflow_datasets,
        force=force_download,
        model_format="yolov8",
    )
    print("Download dos conjuntos de dados do Roboflow concluído com sucesso.")

download_roboflow_data()

Iniciando o processo de download dos conjuntos de dados do Roboflow...
datasets: inside-bus-view já está baixado em /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/raw/inside-bus-view
datasets: passenger-deakin já está baixado em /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/raw/passenger-deakin
Download dos conjuntos de dados do Roboflow concluído com sucesso.


## 1.2 Extração do Hugging Face (CrowdHuman)

Realiza o download dos arquivos originais do dataset CrowdHuman (arquivos de anotação `.odgt` e arquivos compactados `.zip`). 
O parâmetro `fast_mode` permite o download parcial do dataset para agilizar testes de integração de código.

In [3]:
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    raise ImportError("Por favor instale huggingface_hub: uv add huggingface_hub")

def download_crowdhuman_raw(raw_dir: Path, fast_mode: bool = False) -> None:
    f"""
    Baixa as anotações ODGT brutas e os arquivos ZIP do Hugging Face.
    Salva os arquivos diretamente no diretório de dados brutos sem qualquer extração ou processamento.
    """
    repo_id: str = "sshao0516/CrowdHuman"
    target_dir: Path = raw_dir / "CrowdHuman"

    target_dir.mkdir(parents=True, exist_ok=True)

    print(f"Baixando anotações do CrowdHuman para {target_dir}...")
    hf_hub_download(
        repo_id=repo_id,
        filename="annotation_train.odgt",
        repo_type="dataset",
        local_dir=str(target_dir)
    )
    hf_hub_download(
        repo_id=repo_id,
        filename="annotation_val.odgt",
        repo_type="dataset",
        local_dir=str(target_dir)
    )

    print("Baixando arquivos de imagem do CrowdHuman...")
    zips_to_download: List[str] = [
        "CrowdHuman_val.zip",
        "CrowdHuman_train01.zip",
    ]

    if not fast_mode:
        zips_to_download.extend([
            "CrowdHuman_train02.zip",
            "CrowdHuman_train03.zip",
        ])

    for z_name in zips_to_download:
        print(f"Baixando {z_name}...")
        hf_hub_download(
            repo_id=repo_id,
            filename=z_name,
            repo_type="dataset",
            local_dir=str(target_dir)
        )

    print("Download dos dados brutos do CrowdHuman concluído.")

# Para reduzir volume de download em testes rápidos, use `fast_mode=True`.
download_crowdhuman_raw(config.RAW_DIR, fast_mode=False)

Baixando anotações do CrowdHuman para /home/ruben/Projeto/IA901-2026S1/projetos/bus-passenger-count/data/raw/CrowdHuman...
Baixando arquivos de imagem do CrowdHuman...
Baixando CrowdHuman_val.zip...
Baixando CrowdHuman_train01.zip...
Download dos dados brutos do CrowdHuman concluído.
